## 문제 3-1 : 콘텐츠분쟁해결 RAG 시스템

콘텐츠분쟁해결 사례집(`data/콘텐츠분쟁해결_사례.pdf`)을 활용한 법률 자문 RAG 시스템을 순차적으로 구축하는 실습입니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("UPSTAGE_API_KEY 설정 확인 완료")

### 0단계: 문서 로드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

print("0단계 : 문서 로드")
loader = PyPDFLoader('../data/콘텐츠분쟁해결_사례.pdf')
documents = loader.load()

print(f"총 {len(documents)}페이지 로드 완료\n")
print(documents[10].page_content[:300])

### 1단계: 문서 분할 설정

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("1단계 : 문서 분할 설정")
pythontext_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,        # 법률 사례는 1200-1800자 권장
    chunk_overlap=300,      # 사례 맥락 보존을 위해 200-400자
    separators=[
        "\n【사건개요",       # 법률 문서 섹션 구분자 
        "\n【쟁점사항",       # 쟁점 부분 구분
        "\n【처리경위",       # 처리 과정 구분
        "\n【처리결과",       # 결과 부분 구분
        "\n■", "\n\n", "\n", ".", " ", ""
    ]
)
chunks = pythontext_splitter.split_documents(documents)

def is_toc_chunk(text: str, ratio: float = 0.5) -> bool:
    """점선(···)이 들어간 목차 줄이 ratio 이상이면 목차 청크로 판단합니다."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return True 
    toc_lines = sum("···" in line for line in lines)
    return toc_lines / len(lines) >= ratio

body_chunks = [c for c in chunks if not is_toc_chunk(c.page_content)]
print(f"{len(chunks)}개 청크 생성 (목차 {len(chunks) - len(body_chunks)}개 제외 → {len(body_chunks)}개)")
print(f"평균 청크 길이: {sum(len(c.page_content) for c in body_chunks) / len(body_chunks):.0f}자")

### 2단계: 임베딩 모델 설정 및 FAISS 벡터스토어 저장

In [ ]:
from pathlib import Path
from langchain_community.vectorstores import FAISS
from langchain_upstage import UpstageEmbeddings

DB_PATH = "../db/faiss_db_content_dispute"

print("2단계 : 임베딩 모델 설정 및 FAISS 벡터스토어 저장")

pythonembeddings = UpstageEmbeddings(model="solar-embedding-1-large")

if (Path(DB_PATH) / "index.faiss").exists():
    vectorstore = FAISS.load_local(DB_PATH, pythonembeddings, allow_dangerous_deserialization=True)
    print(f"기존 인덱스 로드: {DB_PATH}")
else:
    vectorstore = FAISS.from_documents(body_chunks, pythonembeddings)
    vectorstore.save_local(DB_PATH)
    print(f"FAISS 벡터스토어 생성 및 저장 완료: {DB_PATH} ({len(body_chunks)}개 벡터)")

### 3단계: 검색기 설정

In [ ]:
print("3단계 : 검색기")
pythonretriever = vectorstore.as_retriever(
    search_type="similarity",        # 또는 "mmr" (다양성 고려시)
    search_kwargs={"k": 5}           # 상위 5개 관련 사례 검색
)
print("Retriever 설정 완료\n")

docs = pythonretriever.invoke("게임 아이템 복구")
for i, doc in enumerate(docs, 1):
    print(f"{i}. P{doc.metadata.get('page_label', '?')}: {doc.page_content[:60].replace(chr(10), ' ')}...")

### 4단계: LLM 설정

In [ ]:
from langchain_upstage import ChatUpstage

print("4단계 : LLM 설정")

pythonllm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.2
    )
print(pythonllm.model_name)

### 5단계: 법률 자문 프롬프트 작성

In [ ]:
from langchain_core.prompts import PromptTemplate

print("5단계 : 법률 자문 프롬프트 작성")

def format_docs(docs):
    """검색된 사례들을 출처 페이지와 함께 하나의 문자열로 합칩니다."""
    return "\n\n".join(
        f"[사례 {i} - P{doc.metadata.get('page_label', '?')}]\n{doc.page_content}"
        for i, doc in enumerate(docs, 1)
    )

pythonprompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다. 
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요                    # 사례 기반 답변
2. 관련 법령이나 조항이 있다면 명시하세요               # 법적 근거 제시
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요    # 판례 참조
4. 실무적 해결방안을 단계별로 제시하세요               #  실무 가이드
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요  #  한계 인정

전문 법률 조언:"""

prompt = PromptTemplate.from_template(pythonprompt_template)
print(prompt.input_variables)

### 6단계: LCEL 체인 생성

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("6단계 : LCEL 체인 생성")

pythonqa_chain = (
    {"context": pythonretriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | pythonllm
    | StrOutputParser()
)

def rag_with_sources(question):
    docs = pythonretriever.invoke(question)
    context = format_docs(docs)
    answer = (prompt | pythonllm | StrOutputParser()).invoke({"context": context, "question": question})
    return {"answer": answer, "source_documents": docs}

print("LCEL RAG 파이프라인 구축 완료")

### 8단계: 분쟁 유형 분류 함수 (선택)

> 7단계에서 사용하므로 먼저 실행합니다.

In [ ]:
print("8단계 : 분쟁 유형 분류 함수 (선택)")

def classify_dispute_type(query):
    game_keywords = ["게임", "아이템", "계정", "캐릭터", "레벨", "길드", "온라인게임"]
    elearning_keywords = ["강의", "온라인교육", "이러닝", "수강", "환불", "화상교육"]
    web_keywords = ["웹사이트", "무료체험", "자동결제", "구독", "사이트"]

    query_lower = query.lower()

    if any(keyword in query_lower for keyword in game_keywords):
        return "게임"
    elif any(keyword in query_lower for keyword in elearning_keywords):
        return "이러닝"
    elif any(keyword in query_lower for keyword in web_keywords):
        return "웹콘텐츠"
    else:
        return "기타"

### 7단계: 테스트 질문 작성 

In [ ]:
print("7단계 : 테스트 질문 작성")

pythontest_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?"
]

print("\n" + "=" * 60)
print(" 콘텐츠분쟁해결 RAG 시스템 테스트")
print("=" * 60)

for i, question in enumerate(pythontest_questions, 1):
    print(f"\n[ 테스트 {i}/{len(pythontest_questions)} — 분쟁 유형: {classify_dispute_type(question)} ]")
    print(f"Q: {question}")

    result = rag_with_sources(question)
    print(f"A: {result['answer']}")

    docs = result["source_documents"]
    print(f"\n참조: {len(docs)}개 사례")
    for j, doc in enumerate(docs[:3], 1):
        page = doc.metadata.get("page_label", "?")
        print(f"{j}. P{page}: {doc.page_content[:80].replace(chr(10), ' ')}...")
    print("\n" + "-" * 60)